# CPV VisDrone Training

**Phase 5 sanity:** `EPOCHS = 5`, `CONFIG = yolov8n.yaml`  
**Phase 6 full:** `EPOCHS = 50`, change `CONFIG` per model run

Inputs: `itsdreowo/cpv-visdrone5` (data), `itsdreowo/cpv-source` (code)  
No internet required.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CONFIG    = "yolov8n.yaml"   # yolov8m.yaml | rtdetr.yaml for Phase 6
EPOCHS    = 5                # 5 = sanity, 50 = full
DATA_ROOT = "/kaggle/input/cpv-visdrone5"
DEVICE    = "cuda"

SOURCE    = "/kaggle/input/cpv-source"
WORK      = "/kaggle/working"

In [ ]:
import subprocess, shutil, sys, os
from pathlib import Path

# Copy configs to working dir so relative paths inside YAMLs resolve correctly
shutil.copytree(f"{SOURCE}/configs", f"{WORK}/configs", dirs_exist_ok=True)

os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "filterpy>=1.4.5", "-q"], check=True)
print("Setup complete. CWD:", os.getcwd())

In [ ]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Train
subprocess.run(
    [
        sys.executable, f"{SOURCE}/scripts/train.py",
        "--config",    f"configs/{CONFIG}",
        "--epochs",    str(EPOCHS),
        "--device",    DEVICE,
        "--data-root", DATA_ROOT,
    ],
    check=True,
)

In [ ]:
# Show training curves
import glob
from IPython.display import Image, display

for img in sorted(glob.glob("runs/*/results.png")):
    print(img)
    display(Image(img))

In [ ]:
# Copy best weights to notebook output
for pt in sorted(Path("runs").glob("*/weights/best.pt")):
    dest = Path(WORK) / pt.parent.parent.name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, dest)
    print("Saved:", dest)